In [ ]:
from experiments.lib.reproducibility import Experiment
from mlflow.entities import Run
from pathlib import Path
from typing import Optional
from tempfile import TemporaryDirectory
from contextlib import ExitStack
import mlflow
from experiments.experiment.models.convnext import ConvNext, ConvNextConfig
from experiments.experiment.datamodules.nuclear_cataract_datamodule import (
    NuclearCataractDataModuleConfig,
)
import matplotlib.pyplot as plt
from pytorch_grad_cam import LayerCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

In [ ]:
def find_run(name: str):
    for run in exp.client.search_runs(
        [exp.mlflow_experiment.experiment_id], max_results=9999
    ):
        if (
            "optuna_study" in run.data.tags
            and "mlflow.parentRunId" not in run.data.tags
        ):
            return run

In [ ]:
def find_snapshot(exp: Experiment, run: Run) -> Optional[Path]:
    for art in exp.client.list_artifacts(run.info.run_id):
        if art.path == "model":
            return Path(art.path)

    return None

In [ ]:
def find_config(exp: Experiment, run: Run) -> Optional[Path]:
    for art in exp.client.list_artifacts(run.info.run_id):
        if art.path == "study_config.json":
            return Path(art.path)

    return None

In [ ]:
experiment_name = "augmentations"
run_name = "augmentations_R_6ea4ecb"

exp = Experiment(experiment_name=experiment_name, set_active=False)
run = find_run(run_name)
art = find_snapshot(exp, run)

In [ ]:
if art:
    print("Found model snapshot")
else:
    print("Model snapshot missing")

In [ ]:
cleanup_stack = ExitStack()

In [ ]:
temp_dir = cleanup_stack.enter_context(TemporaryDirectory(prefix="camstudy"))
mlflow.artifacts.download_artifacts(
    run_id=run.info.run_id,
    artifact_path=str(art),
    dst_path=temp_dir,
)

In [ ]:
model = ConvNext.load_from_checkpoint(
    Path(temp_dir) / "model" / "best_retrain.ckpt",
    config=ConvNextConfig(),
    strict=False,
)

In [ ]:
datamodule = NuclearCataractDataModuleConfig().build()
datamodule.setup("train")
data_loader = datamodule.train_dataloader()

img, labels = next(iter(data_loader))

In [ ]:
tested_img_idx = 3
plt.imshow(img[tested_img_idx].permute(1, 2, 0))
plt.show()

In [ ]:
model

In [ ]:
last_conv_layer = model.model.features[-1][-1].block[0]
target_layers = [
    last_conv_layer,
    #    model.model.features[-1][-2].block[0],
    #    model.model.features[-1][-3].block[0],
]
input_tensor = img[tested_img_idx].unsqueeze(0)
targets = [ClassifierOutputTarget(0)]

cam_type = LayerCAM

with cam_type(model=model, target_layers=target_layers) as cam:
    grayscale_cam = cam(input_tensor=input_tensor, targets=targets)
    grayscale_cam = grayscale_cam[0, :]
    visualization = show_cam_on_image(
        img[tested_img_idx].permute(1, 2, 0).numpy(), grayscale_cam, use_rgb=True
    )

In [ ]:
plt.imshow(visualization)
plt.show()

In [ ]:
cleanup_stack.close()